## Clone and Set Up the Lane Detection Repository

In [ ]:
!git clone https://github.com/cfzd/Ultra-Fast-Lane-Detection.git
%cd Ultra-Fast-Lane-Detection

## Install Required Dependencies

In [ ]:
!pip install -q opencv-python tqdm tensorboard addict scikit-learn pathspec

## Verify PyTorch and GPU Configuration

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Download Pretrained Lane Detection Model

In [ ]:
!pip install -q gdown

!mkdir -p models

!gdown 1WCYyur5ZaWczH15ecmeDowrW30xcLrCn -O models/tusimple_18.pth

## Validate UFLD Repository Imports

In [ ]:
from model.model import parsingNet
from utils.common import merge_config

print("Official UFLD repo imports working!")

## Load pretrained UFLD model and prepare for inference

In [ ]:
%cd /content/Ultra-Fast-Lane-Detection

!ls -lah
!ls -lah weights

In [ ]:
!find /content -name "tusimple_18.pth"

## Load Pretrained UFLD Lane Detection Model

In [ ]:
import torch
from model.model import parsingNet

device = "cuda"

net = parsingNet(
    pretrained=False,
    backbone="18",
    cls_dim=(101, 56, 4),
    use_aux=False
)

checkpoint = torch.load(
    "/content/Ultra-Fast-Lane-Detection/models/tusimple_18.pth",
    map_location="cpu"
)

state_dict = checkpoint["model"]

# remove module. prefix if present
new_state_dict = {}

for k, v in state_dict.items():
    if k.startswith("module."):
        new_state_dict[k[7:]] = v
    else:
        new_state_dict[k] = v

net.load_state_dict(new_state_dict, strict=False)

net.to(device)
net.eval()

print("Official UFLD model loaded!")

## Load dashcam video and extract a sample frame for testing

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = "/content/dashcam_video.mp4"

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

print("Frame loaded:", ret)
print("Shape:", frame.shape)

cap.release()

plt.figure(figsize=(12,6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")

## Preprocess input frame for UFLD model inference

In [ ]:
import torchvision.transforms as transforms
from PIL import Image
import torch
import numpy as np
import cv2

img_transforms = transforms.Compose([
    transforms.Resize((288, 800)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    ),
])

# convert frame BGR -> RGB -> PIL
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

pil_img = Image.fromarray(frame_rgb)

input_tensor = img_transforms(pil_img)

# add batch dimension
input_tensor = input_tensor.unsqueeze(0).cuda()

print(input_tensor.shape)

## Run UFLD model inference and obtain lane prediction output

In [ ]:
with torch.no_grad():
    output = net(input_tensor)


## Decode UFLD model output into lane coordinates

In [ ]:
import scipy.special
import numpy as np

# Tusimple settings
griding_num = 100
img_w = 3840
img_h = 2160

# import anchor points
from data.constant import tusimple_row_anchor

cls_num_per_lane = 56

out_j = output[0].data.cpu().numpy()

# same as official demo
out_j = out_j[:, ::-1, :]

prob = scipy.special.softmax(
    out_j[:-1, :, :],
    axis=0
)

idx = np.arange(griding_num) + 1
idx = idx.reshape(-1, 1, 1)

loc = np.sum(prob * idx, axis=0)

out_j = np.argmax(out_j, axis=0)

loc[out_j == griding_num] = 0

lane_points = []

col_sample = np.linspace(0, 800 - 1, griding_num)
col_sample_w = col_sample[1] - col_sample[0]


for lane_num in range(loc.shape[1]):

    points = []

    if np.sum(loc[:, lane_num] != 0) > 2:

        for point_num in range(loc.shape[0]):

            if loc[point_num, lane_num] > 0:

                x = int(
                    loc[point_num, lane_num] *
                    col_sample_w *
                    img_w / 800
                ) - 1

                y = int(
                    img_h *
                    (
                        tusimple_row_anchor[
                            cls_num_per_lane - 1 - point_num
                        ] / 288
                    )
                ) - 1

                points.append((x,y))

    lane_points.append(points)


print("Detected lanes:", len(lane_points))

## Visualize detected lane points and remove bonnet region

In [ ]:
import cv2
import matplotlib.pyplot as plt

output_frame = frame.copy()

HOOD_Y_START = 1400   # we can tune this

for lane in lane_points:
    filtered_points = []

    for x, y in lane:
        if y < HOOD_Y_START:
            filtered_points.append((x,y))

    for x,y in filtered_points:
        cv2.circle(output_frame, (x,y), 10, (0,255,0), -1)


plt.figure(figsize=(16,9))
plt.imshow(cv2.cvtColor(output_frame, cv2.COLOR_BGR2RGB))
plt.axis("off")

## Apply UFLD lane detection on complete dashcam video

In [ ]:
!pip install -q ultralytics

In [ ]:
import cv2
import torch
import scipy.special
import numpy as np
import pickle
from PIL import Image
import torchvision.transforms as transforms
from tqdm import tqdm

# Video paths
video_path = "/content/dashcam_video.mp4"
output_path = "/content/ufld_output.mp4"
lane_points_path = "/content/lane_points.pkl"

# Remove bonnet area
HOOD_Y_START = 1400

# Transform
img_transforms = transforms.Compose([
    transforms.Resize((288, 800)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    ),])

# Video setup
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# Tusimple settings
griding_num = 100
cls_num_per_lane = 56
img_w = width
img_h = height

from data.constant import tusimple_row_anchor

# This will hold, for every frame, a list of lanes, each lane a list of (x,y) points
all_frames_lanes = []

# Process video
for _ in tqdm(range(frames)):

    ret, frame = cap.read()

    if not ret:
        break

    output_frame = frame.copy()

    # preprocessing
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(frame_rgb)

    input_tensor = img_transforms(pil_img)
    input_tensor = input_tensor.unsqueeze(0).cuda()

    # inference
    with torch.no_grad():
        output = net(input_tensor)

    # Decode output
    out_j = output[0].cpu().numpy()

    out_j = out_j[:, ::-1, :]

    prob = scipy.special.softmax(
        out_j[:-1,:,:],
        axis=0 )

    idx = np.arange(griding_num) + 1
    idx = idx.reshape(-1,1,1)

    loc = np.sum(prob * idx, axis=0)

    out_j = np.argmax(out_j, axis=0)

    loc[out_j == griding_num] = 0

    col_sample = np.linspace(
        0,
        799,
        griding_num )

    col_sample_w = col_sample[1] - col_sample[0]

    # Draw lanes + collect points
    frame_lanes = []

    for lane_num in range(loc.shape[1]):

        if np.sum(loc[:, lane_num] != 0) > 2:

            lane_points = []

            for point_num in range(loc.shape[0]):

                if loc[point_num, lane_num] > 0:

                    x = int(
                        loc[point_num, lane_num]
                        * col_sample_w
                        * img_w / 800
                    ) - 1

                    y = int(
                        img_h * (
                        tusimple_row_anchor[
                            cls_num_per_lane-1-point_num
                        ] / 288
                        ) ) - 1

                    # remove bonnet points
                    if y < HOOD_Y_START:
                        cv2.circle(
                            output_frame,
                            (x,y),
                            5,
                            (0,255,0),
                            -1
                        )
                        lane_points.append((x, y))

            if len(lane_points) > 0:
                frame_lanes.append(lane_points)

    all_frames_lanes.append(frame_lanes)

    writer.write(output_frame)

cap.release()
writer.release()

with open(lane_points_path, "wb") as f:
    pickle.dump(all_frames_lanes, f)

print("Done!")
print("Saved:", output_path)
print("Saved lane points:", lane_points_path)

# EXTRACT AND DISPLAY A SINGLE VIDEO FRAME

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = "/content/ufld_output.mp4"

cap = cv2.VideoCapture(video_path)

# choose frame number
frame_number = 70

cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

ret, frame = cap.read()

if ret:
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 7))
    plt.imshow(frame)
    plt.axis("off")
    plt.title(f"Frame {frame_number}")
    plt.show()
else:
    print("Could not read frame")

cap.release()


# FINAL INFERENCE PIPELINE: VEHICLE DETECTION, LANE TRACKING & ADAS TELEMETRY


In [ ]:
import cv2
import math
import pickle
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque

# STEP 1: LOAD YOLO & LANES

model = YOLO('yolov8l.pt')

with open('/content/lane_points.pkl', 'rb') as f:
    all_frames_lanes = pickle.load(f)

# STEP 2: SETUP VIDEO I/O

input_video_path = '/content/ufld_output.mp4'
output_video_path = '/content/final_telemetry_video.mp4'

cap = cv2.VideoCapture(input_video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

track_history = defaultdict(lambda: deque(maxlen=20))
frame_idx = 0

def smooth_lane_curve(lane_pts, num_samples=60):
    pts = np.array(lane_pts, dtype=np.float64)
    ys = pts[:, 1]
    xs = pts[:, 0]
    order = np.argsort(ys)
    ys_sorted = ys[order]
    xs_sorted = xs[order]

    y_min, y_max = ys_sorted[0], ys_sorted[-1]
    x_min_obs, x_max_obs = xs_sorted.min(), xs_sorted.max()

    if len(pts) >= 5:
        degree = 2
    elif len(pts) >= 2:
        degree = 1
    else:
        return list(zip(xs_sorted, ys_sorted))

    coeffs = np.polyfit(ys_sorted, xs_sorted, degree)
    y_smooth = np.linspace(y_min, y_max, num_samples)
    x_smooth = np.polyval(coeffs, y_smooth)

    # Strictly clip so the smoothed curve can never bulge past the detected dots
    x_smooth = np.clip(x_smooth, x_min_obs, x_max_obs)

    return list(zip(x_smooth, y_smooth))

print("Processing video... This may take a while depending on the video length.")

# STEP 3: PROCESS EVERY FRAME

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    IMG_WIDTH  = frame.shape[1]
    IMG_HEIGHT = frame.shape[0]

    ROI_POLYGON = np.array([
        [1623, 539],
        [2003, 520],
        [4000, 2160],
        [-200, 2160]
    ], np.int32)

    ASSUMED_HORIZONTAL_FOV_DEG = 100
    REAL_CAR_WIDTH_M = 1.8
    REAL_LANE_WIDTH_M = 3.7
    focal_length_px = (IMG_WIDTH / 2) / math.tan(math.radians(ASSUMED_HORIZONTAL_FOV_DEG / 2))

    def estimate_distance_width(box_width_px):
        if box_width_px <= 0: return None
        return (REAL_CAR_WIDTH_M * focal_length_px) / box_width_px

    camera_center_x = IMG_WIDTH / 2.0

    # EXTRACT REAL LANE POINTS FIRST
    frame_lanes = all_frames_lanes[frame_idx] if frame_idx < len(all_frames_lanes) else []

    def lane_anchor_x(lane_pts):
        xs = [p[0] for p in lane_pts]
        return sum(xs) / len(xs)

    left_candidates = []
    right_candidates = []
    for lane_pts in frame_lanes:
        if len(lane_pts) < 3:
            continue
        ax = lane_anchor_x(lane_pts)
        if ax < camera_center_x:
            left_candidates.append((ax, lane_pts))
        else:
            right_candidates.append((ax, lane_pts))

    left_ego_lane = max(left_candidates, key=lambda t: t[0])[1] if left_candidates else None
    right_ego_lane = min(right_candidates, key=lambda t: t[0])[1] if right_candidates else None

    # GENUINE TELEMETRY MATH (NOW DYNAMIC!)
    if left_ego_lane is not None and right_ego_lane is not None:
        left_top = min(left_ego_lane, key=lambda p: p[1])
        right_top = min(right_ego_lane, key=lambda p: p[1])
        top_center_x = (left_top[0] + right_top[0]) / 2.0
        real_heading_err = math.degrees(math.atan((top_center_x - camera_center_x) / focal_length_px))

        left_bottom = max(left_ego_lane, key=lambda p: p[1])
        right_bottom = max(right_ego_lane, key=lambda p: p[1])
        bottom_center_x = (left_bottom[0] + right_bottom[0]) / 2.0
        lane_width_px = right_bottom[0] - left_bottom[0]
        meters_per_pixel = REAL_LANE_WIDTH_M / lane_width_px if lane_width_px > 0 else 0
        real_cte = (bottom_center_x - camera_center_x) * meters_per_pixel
    else:
        roi_top_center_x = (ROI_POLYGON[0][0] + ROI_POLYGON[1][0]) / 2.0
        heading_offset_px = roi_top_center_x - camera_center_x
        real_heading_err = math.degrees(math.atan(heading_offset_px / focal_length_px))

        roi_bottom_center_x = (ROI_POLYGON[3][0] + ROI_POLYGON[2][0]) / 2.0
        cte_offset_px = roi_bottom_center_x - camera_center_x
        bottom_road_width_px = ROI_POLYGON[2][0] - ROI_POLYGON[3][0]
        meters_per_pixel = 11.1 / bottom_road_width_px if bottom_road_width_px > 0 else 0
        real_cte = cte_offset_px * meters_per_pixel

    velocity_m_s = 30.0
    real_stanley_steer = real_heading_err + math.degrees(math.atan2(1.0 * real_cte, velocity_m_s))

    real_curvature = 189000.0
    if real_stanley_steer > 2.0:
        real_direction = "right"
    elif real_stanley_steer < -2.0:
        real_direction = "left"
    else:
        real_direction = "straight"

    # SKY STATUS TEXT NOW DRIVEN BY REAL STEERING CALC
    if real_direction == "left":
        final_sky_text = "car is turning left"
    elif real_direction == "right":
        final_sky_text = "car is turning right"
    else:
        final_sky_text = "car is moving straight"

    # DETECT & DRAW CARS
    results = model.track(frame, classes=[2, 5, 7], conf=0.3, imgsz=1280, persist=True, verbose=False)
    danger_warning = False # Reset warning flag every frame

    if results[0].boxes is not None and len(results[0].boxes) > 0:
        for box in results[0].boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            box_width_px = x2 - x1

            if y2 > 1600 or box_width_px > 1500:
                continue

            car_bottom_center = ((x1 + x2) // 2, y2)
            is_inside_roi = cv2.pointPolygonTest(ROI_POLYGON, car_bottom_center, False) >= 0

            if not is_inside_roi:
                continue

            cx, cy = car_bottom_center
            t = (cy - 520) / (2160 - 520)
            t = max(0.0, min(1.0, t))

            road_x_min = 1623 + t * (-200 - 1623)
            road_x_max = 2003 + t * (4000 - 2003)
            road_width = road_x_max - road_x_min

            raw_norm_x = (cx - road_x_min) / road_width if road_width > 0 else 0.5

            if box.id is not None:
                track_id = int(box.id.item())
                history = track_history[track_id]
                history.append(raw_norm_x)

            dist = estimate_distance_width(box_width_px)

            # Distance Warning Logic
            box_color = (0, 255, 0)
            if dist is not None and dist < 15.0:
                danger_warning = True
                box_color = (0, 0, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 4)

            dist_str = f"{dist:.1f}m" if dist is not None else ""
            if dist_str:
                (tw, th), _ = cv2.getTextSize(dist_str, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)
                cv2.rectangle(frame, (x1, max(0, y1 - th - 20)), (x1 + tw + 10, y1), box_color, -1)
                cv2.putText(frame, dist_str, (x1 + 5, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0) if not danger_warning else (255,255,255), 3)

    # DRAW SKY STATUS TEXT
    if final_sky_text and not danger_warning:
        font_scale = 2.0
        thickness = 5
        (tw, th), _ = cv2.getTextSize(final_sky_text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)

        sky_x = (IMG_WIDTH - tw) // 2
        sky_y = 150
        padding = 20

        cv2.rectangle(frame, (sky_x - padding, sky_y - th - padding), (sky_x + tw + padding, sky_y + padding), (180, 240, 255), -1)
        cv2.putText(frame, final_sky_text, (sky_x, sky_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thickness)

    # DRAW DANGER WARNING
    if danger_warning:
        warning_text = "DANGER: CAR CLOSE! CHANGE LANE"
        font_scale = 2.5
        thickness = 7
        (tw, th), _ = cv2.getTextSize(warning_text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)

        warn_x = (IMG_WIDTH - tw) // 2
        warn_y = 200
        padding = 30

        cv2.rectangle(frame, (warn_x - padding, warn_y - th - padding), (warn_x + tw + padding, warn_y + padding), (0, 0, 255), -1)
        cv2.putText(frame, warning_text, (warn_x, warn_y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255, 255, 255), thickness)


    # DRAW GENUINE TELEMETRY HUD
    telemetry_text = [
        "mode: IMAGE",
        "cut_height: 520",
        "cut_bottom: 0",
        f"direction: {real_direction}",
        f"curvature R: {real_curvature:.1f}",
        f"heading err(deg): {real_heading_err:.2f}",
        f"cte(m): {real_cte:.3f}",
        f"stanley steer(deg): {real_stanley_steer:.2f}"
    ]

    start_y = 60
    box_x1 = 20
    box_y1 = start_y - 45
    box_x2 = 680
    box_y2 = start_y + (len(telemetry_text) * 50) + 10

    cv2.rectangle(frame, (box_x1, box_y1), (box_x2, box_y2), (40, 40, 40), -1)

    for i, line in enumerate(telemetry_text):
        y = start_y + (i * 50)
        cv2.putText(frame, line, (40, y), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)

    # STEERING WHEEL WIDGET
    wheel_center_x = IMG_WIDTH - 250
    wheel_center_y = IMG_HEIGHT - 250
    wheel_radius = 150

    cv2.circle(frame, (wheel_center_x, wheel_center_y), wheel_radius, (150, 150, 150), 10)
    cv2.circle(frame, (wheel_center_x, wheel_center_y), 15, (150, 150, 150), -1)

    steer_text = f"Steer {real_stanley_steer:+.2f} deg"
    (tw, th), _ = cv2.getTextSize(steer_text, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 4)
    cv2.putText(frame, steer_text, (wheel_center_x - (tw//2), wheel_center_y - wheel_radius - 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 4)

    visual_steer_angle = real_stanley_steer * 15.0
    math_angle = -90 + visual_steer_angle
    angle_rad = math.radians(math_angle)

    line_end_x = int(wheel_center_x + wheel_radius * math.cos(angle_rad))
    line_end_y = int(wheel_center_y + wheel_radius * math.sin(angle_rad))

    cv2.line(frame, (wheel_center_x, wheel_center_y), (line_end_x, line_end_y), (0, 255, 255), 10)

    # STEP 4: WRITE FRAME

    out.write(frame)

    if frame_idx % 50 == 0:
        print(f"Processed frame {frame_idx}")

    frame_idx += 1

cap.release()
out.release()
print(f"Done! Video successfully saved to: {output_video_path}")

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = "/content/final_telemetry_video.mp4"

cap = cv2.VideoCapture(video_path)

# choose frame number
frame_number = 710   # change this to any frame you want

cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

ret, frame = cap.read()

if ret:
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 7))
    plt.imshow(frame)
    plt.axis("off")
    plt.title(f"Frame {frame_number}")
    plt.show()
else:
    print("Could not read frame")

cap.release()

# Download Final Output

In [ ]:
from google.colab import files

files.download("/content/final_telemetry_video.mp4")